# M9.15 · Eşleştirme — şablon, homografi, RANSAC, panorama, korelasyon filtreleri

Örüntü tanımanın en yalın biçimi şablon eşleme ve normalize korelasyon katsayısı; öznitelik eşleştirmeden homografiye (DLT, Hartley normalizasyonu), aykırı değerlere dayanıklı RANSAC ve alıştırma 6'nın panoraması; şablonu her karede yeniden öğrenen ayırt edici korelasyon süzgeçleri (MOSSE → CSRT). Saf Python'da NCC ve 4 noktadan homografi; NumPy'da `cv2.matchTemplate` ile aynı NCC haritası ve tüm hipotezleri tek SVD çağrısında çözen vektörel RANSAC; PyTorch'ta frekans uzayında MOSSE izleyici; en sonda torch'ta toplu RANSAC + `grid_sample` ile panorama.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-15-matching.html](https://m-gonul.github.io/pytorch-lab/m9-15-matching.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_15_matching.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src_ = ns[\"__source__\"]\n    R.test(\"numpy/cv2 kullanılmıyor\", lambda: no_imports(src_, {\"numpy\", \"cv2\", \"scipy\", \"skimage\"}))\n\n    def n():\n        close(need(ns, \"ncc\")([1, 2, 3, 4], [2, 1, 4, 3]), 0.6, what=\"elle örnek\")\n        close(ns[\"ncc\"]([1, 2, 3], [10, 20, 30]), 1.0, what=\"kazanç değişmezliği\")\n        close(ns[\"ncc\"]([1, 2, 3], [7, 8, 9]), 1.0, what=\"ofset değişmezliği\")\n        ensure(ns[\"ncc\"]([1, 1, 1], [1, 2, 3]) == 0, \"sıfır payda → 0\")\n    R.test(\"ncc\", n)\n\n    def mt():\n        img = data.camera()[200:236, 240:280].astype(np.float32)\n        t = img[10:22, 15:25].copy()\n        got = np.array(need(ns, \"match_template\")(img.tolist(), t.tolist()))\n        want = cv2.matchTemplate(img, t, cv2.TM_CCOEFF_NORMED)\n        ensure(got.shape == want.shape, f\"şekil {got.shape}, {want.shape} bekleniyordu\")\n        close(got, want, atol=1e-4, what=\"cv2 TM_CCOEFF_NORMED\")\n        ensure(np.unravel_index(np.argmax(got), got.shape) == (10, 15), \"en iyi eşleşme şablonun kesildiği yer\")\n    R.test(\"match_template = cv2.matchTemplate\", mt)\n\n    def h4():\n        rng = np.random.default_rng(0)\n        for _ in range(5):\n            Ht = np.array([[rng.uniform(0.8, 1.2), rng.uniform(-0.2, 0.2), rng.uniform(-30, 30)], [rng.uniform(-0.2, 0.2), rng.uniform(0.8, 1.2), rng.uniform(-30, 30)], [rng.uniform(-1e-3, 1e-3), rng.uniform(-1e-3, 1e-3), 1]])\n            src = rng.uniform(0, 400, (4, 2))\n            dst = cv2.perspectiveTransform(src[None], Ht)[0]\n            H = np.array(need(ns, \"solve_h4\")(src.tolist(), dst.tolist()))\n            close(H, Ht, atol=1e-6, what=\"homografi\")\n            close(np.array(need(ns, \"apply_h\")(H.tolist(), 10.0, 20.0)), cv2.perspectiveTransform(np.array([[[10.0, 20.0]]]), Ht)[0, 0], atol=1e-6, what=\"apply_h\")\n    R.test(\"solve_h4 ve apply_h\", h4)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef corners_err(H, Ht, size=500):\n    c = np.array([[[0, 0], [size, 0], [size, size], [0, size]]], np.float64)\n    return float(np.abs(cv2.perspectiveTransform(c, H) - cv2.perspectiveTransform(c, Ht)).max())\n\n\ndef check(ns, R):\n    src_ = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src_))\n    img = data.camera().astype(np.float32)\n\n    def nm():\n        for (y, x, h, w) in [(200, 250, 40, 50), (10, 10, 21, 33), (400, 300, 64, 64)]:\n            t = img[y:y + h, x:x + w].copy()\n            got, want = need(ns, \"ncc_map\")(img, t), cv2.matchTemplate(img, t, cv2.TM_CCOEFF_NORMED)\n            ensure(got.shape == want.shape, \"şekil\")\n            tz = t.astype(np.float64) - t.mean()\n            rr = np.random.default_rng(h)\n            for yy, xx in zip(rr.integers(0, got.shape[0], 200), rr.integers(0, got.shape[1], 200)):\n                wz = img[yy:yy + h, xx:xx + w].astype(np.float64); wz = wz - wz.mean()\n                den = np.sqrt((tz ** 2).sum() * (wz ** 2).sum())\n                if den > 1e-3:\n                    close(got[yy, xx], (tz * wz).sum() / den, atol=1e-9, what=f\"({yy}, {xx}) doğrudan tanım\")\n            f64 = img.astype(np.float64)\n            I1 = np.pad(f64, ((1, 0), (1, 0))).cumsum(0).cumsum(1); I2 = np.pad(f64 ** 2, ((1, 0), (1, 0))).cumsum(0).cumsum(1)\n            Sb = lambda I: I[h:, w:] - I[:-h, w:] - I[h:, :-w] + I[:-h, :-w]\n            tex = np.sqrt(np.maximum(Sb(I2) / (h * w) - (Sb(I1) / (h * w)) ** 2, 0)) > 5\n            close(got[tex], want[tex], atol=5e-3, what=f\"şablon {h} × {w}, dokulu pencerelerde cv2\")\n            ensure(np.unravel_index(np.argmax(got), got.shape) == (y, x), \"en iyi konum\")\n    R.test(\"ncc_map = cv2.matchTemplate\", nm)\n\n    rng = np.random.default_rng(0)\n    Ht = np.array([[0.9, 0.1, 30], [-0.05, 1.1, -20], [1e-4, 2e-4, 1]])\n\n    def d():\n        src = rng.uniform(0, 500, (30, 2))\n        dst = cv2.perspectiveTransform(src[None], Ht)[0]\n        q, T = need(ns, \"normalize_points\")(src)\n        close(q.mean(0), [0, 0], atol=1e-9, what=\"ağırlık merkezi\"); close(np.linalg.norm(q, axis=1).mean(), np.sqrt(2), what=\"ortalama uzaklık\")\n        close(need(ns, \"dlt\")(src, dst), Ht, atol=1e-8, what=\"gürültüsüz DLT\")\n        noisy = dst + rng.normal(0, 0.5, dst.shape)\n        Hc, _ = cv2.findHomography(src, noisy, 0)\n        H = ns[\"dlt\"](src, noisy)\n        ensure(corners_err(H, Ht) < corners_err(Hc, Ht) + 1.0, \"gürültülü DLT OpenCV ile benzer doğrulukta olmalı\")\n    R.test(\"normalize_points ve dlt\", d)\n\n    def r():\n        rows = []\n        for ratio in (0.2, 0.4, 0.6):\n            src = rng.uniform(0, 500, (200, 2))\n            dst = cv2.perspectiveTransform(src[None], Ht)[0] + rng.normal(0, 0.5, (200, 2))\n            out = rng.random(200) < ratio\n            dst[out] = rng.uniform(0, 500, (out.sum(), 2))\n            H, m = need(ns, \"ransac\")(src, dst, 3.0, 1000)\n            Hc, mc = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)\n            e, ec = corners_err(H, Ht), corners_err(Hc, Ht)\n            ensure(e < 3 and (m == ~out).mean() > 0.95, f\"%{int(ratio * 100)} aykırı: köşe hatası {e:.2f}, maske doğruluğu {(m == ~out).mean():.2f}\")\n            rows.append([f\"%{int(ratio * 100)}\", int(m.sum()), int((~out).sum()), f\"{e:.2f}\", f\"{ec:.2f}\"])\n        R.table(\"200 eş, σ = 0.5 gürültü\", [\"aykırı\", \"bulunan içerdeğer\", \"gerçek içerdeğer\", \"köşe hatası (px)\", \"cv2 köşe hatası\"], rows)\n    R.test(\"ransac (tüm hipotezler tek SVD'de)\", r)\n",
 "d": "import math\nimport numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef make_video(T=60, seed=0, S=64, rot=2.0):\n    rng = np.random.default_rng(seed)\n    bg = cv2.resize(data.moon(), (320, 240)).astype(np.float32)\n    obj = data.camera()[180:180 + S, 200:200 + S].astype(np.float32)\n    frames, centers = [], []\n    cx, cy, vx, vy = 80.0, 80.0, 2.0, 1.2\n    for t in range(T):\n        f = bg.copy()\n        x0, y0 = int(round(cx - S / 2)), int(round(cy - S / 2))\n        M = cv2.getRotationMatrix2D((S / 2, S / 2), t * rot, 1.0)\n        f[y0:y0 + S, x0:x0 + S] = cv2.warpAffine(obj, M, (S, S), borderMode=cv2.BORDER_REFLECT)\n        frames.append(np.clip(f + rng.normal(0, 8, f.shape), 0, 255).astype(np.float32))\n        centers.append((x0 + S / 2, y0 + S / 2))\n        cx += vx + rng.normal(0, 0.5); cy += vy + math.sin(t / 6) * 1.5\n        if not 40 < cx < 280:\n            vx = -vx\n        if not 40 < cy < 200:\n            vy = -vy\n    return frames, centers\n\n\ndef check(ns, R):\n    def pre():\n        p = np.random.default_rng(1).random((32, 48)) * 255\n        q = need(ns, \"preprocess\")(p).numpy()\n        z = np.log(p + 1); z = (z - z.mean()) / (z.std() + 1e-5)\n        close(q, z * np.outer(np.hanning(32), np.hanning(48)), atol=1e-12, what=\"ön işleme\")\n        g = torch.fft.ifft2(need(ns, \"target\")(32, 48)).real\n        ensure(tuple(np.unravel_index(int(g.argmax()), g.shape)) == (16, 24), \"Gauss tepe merkezde\")\n    R.test(\"preprocess ve target\", pre)\n\n    def track():\n        fr, c = make_video()\n        tr = need(ns, \"Mosse\", \"sınıf\")(fr[0], *c[0])\n        errs, psrs = [], []\n        for f, cc in zip(fr[1:], c[1:]):\n            x, y, p = tr.update(f)\n            errs.append(math.hypot(x - cc[0], y - cc[1])); psrs.append(p)\n        tm = cv2.getRectSubPix(fr[0], (64, 64), c[0])\n        e2 = []\n        for f, cc in zip(fr[1:], c[1:]):\n            r = cv2.matchTemplate(f, tm, cv2.TM_CCOEFF_NORMED); y, x = np.unravel_index(r.argmax(), r.shape)\n            e2.append(math.hypot(x + 32 - cc[0], y + 32 - cc[1]))\n        ensure(np.mean(errs) < 6 and max(errs) < 15, f\"MOSSE ortalama hata {np.mean(errs):.1f}, en büyük {max(errs):.1f}\")\n        ensure(np.mean(e2) > 20, \"sabit şablon dönen hedefte kaymalı\")\n        R.table(\"60 kare, hedef her karede 2° dönüyor, σ = 8 gürültü\", [\"yöntem\", \"ortalama merkez hatası\", \"en büyük\"],\n                [[\"sabit şablon NCC\", f\"{np.mean(e2):.1f}\", f\"{max(e2):.1f}\"], [\"MOSSE (η = 0.125)\", f\"{np.mean(errs):.1f}\", f\"{max(errs):.1f}\"]],\n                note=f\"MOSSE PSR: ortalama {np.mean(psrs):.1f}, en düşük {min(psrs):.1f} (Bolme: PSR < 7 izleme kaybı işareti).\")\n    R.test(\"dönen hedefi izle\", track)\n",
 "e": "import time\nimport numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import need, ensure, bench, fmt_time\n\n\ndef views():\n    big = data.astronaut()\n    H1 = np.array([[1.0, 0, 0], [0, 1.0, 0], [0, 0, 1]])\n    A = np.array([[0.97, -0.06, 0], [0.05, 0.98, 0], [1.2e-4, -0.8e-4, 1]])\n    T = np.array([[1, 0, -220], [0, 1, -40], [0, 0, 1]])\n    H2 = A @ T\n    v1 = cv2.warpPerspective(big, H1, (320, 360))\n    v2 = cv2.warpPerspective(big, H2, (320, 360))\n    return big, v1, v2, H1 @ np.linalg.inv(H2)\n\n\ndef check(ns, R):\n    big, v1, v2, Ht = views()\n\n    def rs():\n        rng = np.random.default_rng(0)\n        src = rng.uniform(0, 500, (300, 2))\n        dst = cv2.perspectiveTransform(src[None], Ht)[0] + rng.normal(0, 0.5, (300, 2))\n        out = rng.random(300) < 0.5\n        dst[out] = rng.uniform(0, 500, (out.sum(), 2))\n        H, m = need(ns, \"ransac_t\")(src, dst, 3.0, 1000)\n        c = np.array([[[0, 0], [320, 0], [320, 360], [0, 360]]], np.float64)\n        e = np.abs(cv2.perspectiveTransform(c, H.numpy()) - cv2.perspectiveTransform(c, Ht)).max()\n        ensure(e < 3 and (m.numpy() == ~out).mean() > 0.95, f\"köşe hatası {e:.2f}\")\n        t1 = bench(lambda: ns[\"ransac_t\"](src, dst, 3.0, 1000), repeat=3)\n        t2 = bench(lambda: cv2.findHomography(src, dst, cv2.RANSAC, 3.0), repeat=3)\n        R.table(\"%50 aykırı, 300 eş\", [\"yöntem\", \"süre\"], [[\"ransac_t (1000 hipotez tek SVD'de)\", fmt_time(t1)], [\"cv2.findHomography RANSAC (uyarlamalı iterasyon)\", fmt_time(t2)]])\n    R.test(\"ransac_t\", rs)\n\n    def st():\n        pano, H = need(ns, \"stitch\")(v1, v2)\n        c = np.array([[[0, 0], [319, 0], [319, 359], [0, 359]]], np.float64)\n        e = np.abs(cv2.perspectiveTransform(c, H.numpy()) - cv2.perspectiveTransform(c, Ht)).max()\n        ensure(e < 2.0, f\"köşe hatası {e:.2f} piksel\")\n        ensure(pano.dtype == np.uint8 and pano.shape[1] > 450, f\"panorama boyutu {pano.shape}\")\n        ov = big[60:300, 230:300].astype(float)\n        P = pano.astype(float)\n        best = max(10 * np.log10(255 ** 2 / ((P[60 + dy:300 + dy, 230 + dx:300 + dx] - ov) ** 2).mean()) for dy in range(-2, 3) for dx in range(-2, 3))\n        ensure(best > 25, f\"örtüşme bölgesi PSNR {best:.1f}\")\n        return f\"panorama {pano.shape[1]} × {pano.shape[0]}, homografi köşe hatası {e:.2f} px, örtüşmede PSNR {best:.1f} dB\"\n    R.test(\"stitch (alıştırma 6)\", st)\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    img = data.camera().astype(np.float32)\n    locs = [(250, 200), (120, 380), (400, 60), (330, 300)]\n    names = {cv2.TM_SQDIFF: \"SQDIFF\", cv2.TM_SQDIFF_NORMED: \"SQDIFF_NORMED\", cv2.TM_CCORR: \"CCORR\", cv2.TM_CCORR_NORMED: \"CCORR_NORMED\", cv2.TM_CCOEFF: \"CCOEFF\", cv2.TM_CCOEFF_NORMED: \"CCOEFF_NORMED\"}\n\n    def modes():\n        rows = []\n        for m, n in names.items():\n            ok = 0\n            for x, y in locs:\n                t = img[y:y + 40, x:x + 50].copy()\n                for g, o in [(1.0, 0), (0.6, 60), (1.3, -30)]:\n                    search = img.copy(); search[y:y + 40, x:x + 50] = np.clip(t * g + o, 0, 255)\n                    ok += need(ns, \"best_match\")(search, t, m) == (x, y)\n            rows.append([n, f\"{ok}/12\"])\n        d = dict((r[0], int(r[1].split(\"/\")[0])) for r in rows)\n        ensure(d[\"CCOEFF_NORMED\"] == 12, \"korelasyon katsayısı her durumda bulmalı\")\n        R.table(\"Hedef bölgeye kazanç/ofset uygulanınca doğru bulma\", [\"yöntem\", \"doğru\"], rows, note=\"Ham CCORR parlak bölgelere, SQDIFF ofsete takılır; yalnız ortalaması çıkarılmış ve normalize edilmiş ölçü hepsini bulur.\")\n    R.test(\"eşleme yöntemleri\", modes)\n\n    def ms():\n        t = img[200:248, 260:320].copy()\n        small = cv2.resize(img, None, fx=0.7, fy=0.7, interpolation=cv2.INTER_AREA)\n        x, y, s, sc = need(ns, \"multiscale\")(small, t, np.arange(0.5, 1.01, 0.05))\n        ensure(abs(s - 0.7) < 0.051 and abs(x - 260 * 0.7) <= 3 and abs(y - 200 * 0.7) <= 3, f\"bulunan ({x}, {y}), ölçek {s:.2f}\")\n        return f\"ölçek {s:.2f}, skor {sc:.3f}\"\n    R.test(\"çok ölçekli eşleme\", ms)\n",
 "x2": "import numpy as np\nimport cv2\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n    Ht = np.array([[1.02, 0.03, 40], [-0.02, 0.98, 25], [2e-5, -1e-5, 1]])\n\n    def t():\n        er, en, cr, cn = [], [], [], []\n        for k in range(200):\n            src = rng.uniform(0, 1000, (8, 2))\n            dst = cv2.perspectiveTransform(src[None], Ht)[0] + rng.normal(0, 1.0, (8, 2))\n            test = rng.uniform(0, 1000, (100, 2))\n            want = cv2.perspectiveTransform(test[None], Ht)[0]\n            for H, acc in ((need(ns, \"dlt_raw\")(src, dst), er), (need(ns, \"dlt_norm\")(src, dst), en)):\n                acc.append(np.linalg.norm(cv2.perspectiveTransform(test[None], H)[0] - want, axis=1).mean())\n            cr.append(need(ns, \"cond_A\")(src, dst, False)); cn.append(ns[\"cond_A\"](src, dst, True))\n        ensure(np.median(cn) * 1e4 < np.median(cr), \"normalizasyon koşul sayısını büyük ölçüde düşürmeli\")\n        ensure(np.mean(en) < np.mean(er), \"normalize DLT görülmemiş noktalarda daha doğru olmalı\")\n        R.table(\"200 deneme, 8 eş, σ = 1 piksel, koordinatlar 0–1000\", [\"DLT\", \"A'nın koşul sayısı (medyan)\", \"görülmemiş noktalarda hata (px)\"],\n                [[\"normalizasyonsuz\", f\"{np.median(cr):.1e}\", f\"{np.mean(er):.2f}\"], [\"Hartley normalizasyonlu\", f\"{np.median(cn):.1e}\", f\"{np.mean(en):.2f}\"]])\n    R.test(\"normalizasyonun etkisi\", t)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data, color\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def rec():\n        for x in (data.camera().astype(np.float64), data.coins().astype(np.float64)[:301, :383]):\n            close(need(ns, \"collapse\")(need(ns, \"laplacian_pyramid\")(x, 5)), x, atol=1e-9, what=f\"{x.shape} yeniden kurma\")\n        G = need(ns, \"gaussian_pyramid\")(data.camera(), 4)\n        ensure([g.shape for g in G] == [(512, 512), (256, 256), (128, 128), (64, 64)], \"piramit boyutları\")\n    R.test(\"Laplacian piramidi kayıpsız\", rec)\n\n    def bl():\n        a = data.camera().astype(np.float64)\n        b = cv2.resize(color.rgb2gray(data.astronaut()) * 255, (512, 512))\n        b = b * 0.7 + 60\n        mask = np.zeros((512, 512)); mask[:, :256] = 1\n        hard = a * mask + b * (1 - mask)\n        soft = need(ns, \"blend\")(a, b, mask, 6)\n        jump = lambda im: np.abs(im[:, 257] - im[:, 254]).mean()\n        ensure(jump(soft) < jump(hard) * 0.6, f\"dikişteki sıçrama: sert {jump(hard):.1f}, harmanlanmış {jump(soft):.1f}\")\n        far = np.abs(soft[:, :200] - a[:, :200]).mean()\n        ensure(far < 3, \"dikişten uzakta görüntü korunmalı\")\n        return f\"dikişte ortalama sıçrama: sert kesim {jump(hard):.1f}, çok bantlı {jump(soft):.1f}; dikişten uzakta değişim {far:.2f}\"\n    R.test(\"çok bantlı harmanlama\", bl)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_15_matching as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Şablon eşleme:** $w$ şablonunu $f$ üzerinde kaydır. Ham korelasyon $\sum w\,f$ parlak bölgelerde her zaman büyüktür; bunun yerine **korelasyon katsayısı**

$$\gamma(x,y)=\frac{\sum_{s,t}[w(s,t)-\bar w]\,[f(x+s,y+t)-\bar f_{xy}]}{\sqrt{\sum[w-\bar w]^2\sum[f-\bar f_{xy}]^2}}\in[-1,1]$$

genlik (kazanç) ve ofset değişimine değişmezdir. En büyük $\gamma$ en iyi eşleşmedir. Dönme ve ölçek için şablonun dönmüş/ölçeklenmiş kopyaları gerekir; ya da SIFT gibi değişmez öznitelikler (M9.14) eşleştirilir.

**Homografi:** düzlemsel sahne ya da yalnız dönen kamera için iki görüntü arasındaki dönüşüm $\tilde x'=H\tilde x$ ($3\times3$, ölçek serbest → 8 serbestlik). Her nokta eşi iki doğrusal denklem verir: **DLT** $A\mathbf h=0$, $\mathbf h$ = $A$'nın en küçük tekil değerine ait sağ tekil vektör; 4 nokta yeter, fazlası en küçük kareler. **Hartley normalizasyonu:** noktaları ağırlık merkezi 0, ortalama uzaklık $\sqrt2$ olacak şekilde ölçekle; aksi halde $A$'nın sütunları $10^6$ kat farklı ölçeklerde olur, çözüm kararsızlaşır.

**RANSAC:** eşleşmelerin önemli bir kısmı yanlıştır. Tekrarla: rastgele 4 eş, $H$, eşik içinde kalan **içerdeğerleri** say; en çok içerdeğerli hipotezi seç ve yalnız onlarla yeniden kestir. Gereken iterasyon $N=\lceil\log(1-p)/\log(1-w^s)\rceil$.

**Korelasyon süzgeçleri (DCF):** şablonu sabit tutmak yerine, hedef üzerinde istenen yanıtı (merkezde Gauss tepe) veren süzgeci **öğren**. Frekans uzayında kapalı çözüm (MOSSE): $H^*=\frac{\sum_iG_i\odot F_i^*}{\sum_iF_i\odot F_i^*+\lambda}$; her karede $A_t=\eta GF^*+(1-\eta)A_{t-1}$, $B_t=\eta FF^*+(1-\eta)B_{t-1}$ ile güncelle. Ön işleme: log, normalize, kosinüs (Hann) penceresi. Güven: tepe/yan lob oranı **PSR** $=\frac{g_{max}-\mu_{yan}}{\sigma_{yan}}$. Sınırlar: dairesel korelasyon gerçek dışı kaydırılmış örneklerle eğitir, dikdörtgen olmayan hedefte arka planı öğrenir — CSR-DCF (CSRT) uzamsal ve kanal güvenilirliğiyle bunları çözer.

## 1 · Elle

$\gamma$: sıfır ortalamalı vektörler $[-1.5,-0.5,0.5,1.5]$ ve $[-0.5,-1.5,1.5,0.5]$; iç çarpım 3, normlar $\sqrt5\sqrt5=5$.

$N(0.5)=\log0.01/\log(1-1/16)=71.4\to72$; $N(0.8)=\log0.01/\log(1-0.4096)=8.7\to9$: içerdeğer oranı iterasyon sayısını üstel etkiler.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Korelasyon ve homografi**

- `ncc`: $\gamma([1,2,3,4],[2,1,4,3])$
- `npt`: Homografiyi belirlemek için gereken en az nokta eşi
- `hp`: $H=\begin{bmatrix}1&0&0\\0&1&0\\0.01&0&1\end{bmatrix}$ ile $(100,50)$'nin $x'$'i

**RANSAC ve MOSSE**

- `N`: $p=0.99$, içerdeğer oranı $w=0.5$, $s=4$: gereken iterasyon (yukarı yuvarla)
- `N2`: Aynı, $w=0.8$
- `mo`: 1-B MOSSE: $F=2$, $G=4$, $\lambda=1$: $H^*=GF^*/(FF^*+\lambda)$

In [ ]:
cevaplar = dict(
    ncc=None,
    npt=None,
    hp=None,
    N=None,
    N2=None,
    mo=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"ncc":[0.6,0],"npt":[4,0],"hp":[50,0],"N":[72,0],"N2":[9,0],"mo":[1.6,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `ncc(a, b)` — iki düz liste arasında korelasyon katsayısı (paydası 0 ise 0).
- `match_template(img, t)` — "valid" NCC haritası (liste listesi), `cv2.matchTemplate(…, cv2.TM_CCOEFF_NORMED)` ile aynı boyut ve (1e-4 içinde) aynı değerler.
- `solve_h4(src, dst)` — 4 nokta eşinden $h_{33}=1$ alarak 8 × 8 doğrusal sistemi **kısmi pivotlamalı Gauss eliminasyonuyla** çöz; $3\times3$ liste.
- `apply_h(H, x, y)` — homojen koordinatla $(x', y')$.

In [ ]:
%%writefile stage_b.py
import math


def ncc(a, b):
    ...


def match_template(img, t):
    ...


def solve_h4(src, dst):
    ...


def apply_h(H, x, y):
    ...


print(ncc([1, 2, 3, 4], [2, 1, 4, 3]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**.

- `ncc_map(img, t)` — `cv2.matchTemplate(img, t, TM_CCOEFF_NORMED)` ile aynı (OpenCV `float32` DFT ve toplamlar kullandığı için neredeyse düz pencerelerde 0.02'ye varan hata yapar; sen `float64` hesapla — kontrol seni doğrudan tanımla, OpenCV'yi yalnız dokulu pencerelerde karşılaştırır). Pay: sıfır ortalamalı şablonla korelasyon (`scipy.signal.fftconvolve(img, tz[::-1, ::-1], "valid")`); payda: pencere varyansı **integral görüntülerden** ($\sum f$ ve $\sum f^2$). Varyans ≈ 0 olan yerde 0.
- `normalize_points(p)` → `(q, T)`: Hartley; $q=T\tilde p$.
- `dlt(src, dst)` — normalize et, $A$ (2N × 9), `np.linalg.svd`, son satır, geri normalize ($H=T_d^{-1}\hat HT_s$), $h_{33}=1$.
- `ransac(src, dst, thr=3.0, iters=500, seed=0)` → `(H, mask)`: `rng = np.random.default_rng(seed)`, örnekler `rng.random((iters, N)).argsort(1)[:, :4]`; **tüm hipotezleri tek seferde** (yığın `(iters, 8, 9)` → `np.linalg.svd` yığın üzerinde çalışır); hepsinin yeniden izdüşüm hatası `(iters, N)`; en çok içerdeğerli (eşitlikte ilk); `dlt` ile içerdeğerlerden yeniden kestir; son maske bu $H$'ye göre.

In [ ]:
%%writefile stage_c.py
import numpy as np
from scipy import signal


def ncc_map(img, t):
    ...


def normalize_points(p):
    ...


def dlt(src, dst):
    ...


def ransac(src, dst, thr=3.0, iters=500, seed=0):
    ...


rng = np.random.default_rng(0)
s = rng.uniform(0, 100, (10, 2))
print(dlt(s, s + 5))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

MOSSE izleyici (PyTorch, `torch.fft`). Pencere `w × h` (varsayılan 64 × 64), kırpma `cv2.getRectSubPix(frame, (w, h), (cx, cy))`.

- `preprocess(p)` — `log(p + 1)`, ortalama 0 / std 1 (`+1e-5`), `np.outer(np.hanning(h), np.hanning(w))` ile çarp; `float64` tensör.
- `target(h, w, s=2.0)` — merkezi `(h//2, w//2)` olan Gauss yanıtının `fft2`'si.
- `class Mosse(frame, cx, cy, w=64, h=64, lam=1e-2, n_aug=8, lr=0.125, seed=0)` — başlangıç: ilk kırpma + `n_aug − 1` rastgele küçük afin bozulma (`rng = np.random.default_rng(seed)`, açı ∈ [−10°, 10°], ölçek ∈ [0.95, 1.05], `cv2.getRotationMatrix2D((w/2, h/2), …)`, `cv2.warpAffine(…, borderMode=cv2.BORDER_REFLECT)`); $A=\sum GF^*$, $B=\sum FF^*$.
- `update(frame)` → `(cx, cy, psr)`: yanıt `ifft2(F · A/(B+λ)).real`, tepenin merkeze göre ofsetiyle konumu güncelle; PSR: tepe çevresindeki 11 × 11 dışındaki yan lob; yeni konumda kırpıp $A,B$'yi `lr` ile güncelle.

Kontrol, dönen ve hareket eden bir hedef videosunda MOSSE'yi sabit şablonlu NCC ile karşılaştırır.

In [ ]:
%%writefile stage_d.py
import math
import numpy as np
import cv2
import torch


def preprocess(p):
    ...


def target(h, w, s=2.0):
    ...


class Mosse:
    def __init__(self, frame, cx, cy, w=64, h=64, lam=1e-2, n_aug=8, lr=0.125, seed=0):
        ...

    def update(self, frame):
        ...


f = np.random.default_rng(0).random((120, 160)).astype(np.float32) * 255
tr = Mosse(f, 80, 60)
print(tr.update(f))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Panorama, PyTorch'ta toplu RANSAC ile.

- `ransac_t(src, dst, thr=3.0, iters=1000, seed=0)` — c'deki `ransac`'ın torch hali (`torch.Generator().manual_seed(seed)`, `torch.rand(iters, N).argsort(1)[:, :4]`, yığın `torch.linalg.svd`); içerdeğerlerle Hartley normalize DLT. `(H, mask)` (`float64` tensör, bool).
- `stitch(img1, img2)` → `(pano, H)`: `cv2.SIFT_create()`, oran testi 0.75, `ransac_t` ile $H$ (img2 → img1). Tuval: iki görüntünün köşelerini kapsayan dikdörtgen (ötelemeyle negatif koordinatları kaydır); her tuval pikselini `grid_sample` ile her iki görüntüden örnekle (`align_corners=True`, geçerlilik maskesiyle); **tüy (feather) harmanlama**: ağırlık = görüntü kenarına uzaklık (`cv2.distanceTransform` ya da kapalı form), ağırlıklı ortalama. `pano` `uint8`.

In [ ]:
%%writefile stage_e.py
import numpy as np
import cv2
import torch
import torch.nn.functional as F


def ransac_t(src, dst, thr=3.0, iters=1000, seed=0):
    ...


def stitch(img1, img2):
    ...


print("panorama")

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Şablon eşleme yöntemleri ve çok ölçekli arama

- `best_match(img, t, method)` — `cv2.matchTemplate` ile `method` için en iyi konum `(x, y)` (SQDIFF türlerinde en küçük, diğerlerinde en büyük).
- `multiscale(img, t, scales)` — her ölçek için `cv2.resize(t, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)` ile `TM_CCOEFF_NORMED`; en iyi `(x, y, s, score)`.

Kontrol, arama görüntüsüne kazanç/ofset uygulanınca hangi yöntemlerin doğru yeri bulduğunu tablolar.

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def best_match(img, t, method):
    ...


def multiscale(img, t, scales):
    ...


from skimage import data
im = data.camera().astype(np.float32)
print(best_match(im, im[100:140, 200:250].copy(), cv2.TM_CCOEFF_NORMED))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Hartley normalizasyonu neden şart?

- `dlt_raw(src, dst)` — normalizasyonsuz DLT (SVD), $h_{33}=1$.
- `dlt_norm(src, dst)` — Hartley normalizasyonlu DLT.
- `cond_A(src, dst, normalize)` — $A$'nın koşul sayısı (`np.linalg.cond`), normalize edilmiş ya da edilmemiş noktalarla.

Kontrol: 1000 piksellik koordinatlarda σ = 1 gürültüyle 8 eşten kestirim, görülmemiş noktalarda yeniden izdüşüm hatası.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def dlt_raw(src, dst):
    ...


def dlt_norm(src, dst):
    ...


def cond_A(src, dst, normalize):
    ...


s = np.random.default_rng(0).uniform(0, 1000, (8, 2))
print(cond_A(s, s + 3, False), cond_A(s, s + 3, True))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Laplacian piramidiyle dikişsiz harmanlama

Burt & Adelson (1983) çok bantlı harmanlama:

- `gaussian_pyramid(img, levels)` — `cv2.pyrDown` ile; `float64`.
- `laplacian_pyramid(img, levels)` — $L_i=G_i-\mathrm{pyrUp}(G_{i+1})$ (boyut `dstsize` ile eşlenir), son seviye $G_{levels-1}$.
- `collapse(L)` — tersini al: en kabadan başlayıp `pyrUp` + topla. `collapse(laplacian_pyramid(x))` $x$'i (kayan nokta hatası içinde) geri vermeli.
- `blend(a, b, mask, levels=5)` — her seviyede $L^a_i\,G^m_i+L^b_i(1-G^m_i)$ ($G^m$ maskenin Gauss piramidi), sonra `collapse`.

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def gaussian_pyramid(img, levels):
    ...


def laplacian_pyramid(img, levels):
    ...


def collapse(L):
    ...


def blend(a, b, mask, levels=5):
    ...


from skimage import data
x = data.camera().astype(np.float64)
print(np.abs(collapse(laplacian_pyramid(x, 5)) - x).max())

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: CSR-DCF: kanal ve uzamsal güvenilirlikli korelasyon süzgeci

[Lukežič, Vojíř, Čehovin Zajc, Matas, Kristan — Discriminative Correlation Filter with Channel and Spatial Reliability (CVPR 2017)](https://arxiv.org/abs/1611.08461)

d aşamasındaki MOSSE'yi CSRT'ye (OpenCV'deki adıyla) genişlet: HOG + renk adları (color names) öznitelik kanalları, ön plan/arka plan histogramlarından uzamsal güvenilirlik haritası ve bu maskeyle kısıtlı süzgeç öğrenimi (ADMM), kanal başına güvenilirlik ağırlıkları. VOT ya da OTB'den birkaç dizide `cv2.TrackerCSRT_create` ve MOSSE ile karşılaştır.


**Adımlar**

1. Çok kanallı DCF: kanal başına süzgeç, yanıtların ağırlıklı toplamı.
2. Uzamsal güvenilirlik: renk histogramlarından olasılık + Markov rastgele alan düzgünleştirmesi (basitleştirilmiş: morfolojik kapama, M9.8).
3. Maskeli süzgeç için ADMM (makaledeki Algoritma 1).
4. Başarı grafiği (IoU eşiğine karşı başarı oranı) ve hassasiyet grafiği (merkez hatası).

**Kabul ölçütleri**

- En az 5 dizide başarı grafiği; CSR-DCF'in AUC'si MOSSE'den yüksek.
- Uzamsal maskenin kapalı/açık ablasyonu.

## Makale ödevi: Otomatik panorama (alıştırma 6'nın tamamı)

[Brown & Lowe — Automatic Panoramic Image Stitching using Invariant Features (IJCV 2007)](https://doi.org/10.1007/s11263-006-0002-3)

Sırasız bir görüntü kümesinden panorama: SIFT eşleştirme, RANSAC ile homografi, görüntü eşleşme grafiği ve bağlı bileşenler, demet ayarı (bundle adjustment; basitleştirilmiş: dönme + odak uzaklığı), silindirik izdüşüm, kazanç telafisi ve çok bantlı (Laplacian piramidi, x3) harmanlama. Altın Kapı (goldengate-00…05) dizisinde `cv2.Stitcher` ile karşılaştır.


**Adımlar**

1. Her görüntü çifti için eşleşme sayısı ve olasılıksal doğrulama ($n_i>8+0.3n_f$).
2. Silindirik çarpıtma ve yalnız ötelemeli hizalamanın basit sürümü; sonra tam homografi zinciri.
3. Kazanç telafisi: örtüşmelerde ortalama yoğunluk eşitleme (en küçük kareler).

**Kabul ölçütleri**

- 6 görüntülük dizide gözle görülür dikiş ve hayalet olmadan panorama.
- Çift başına RANSAC içerdeğer sayıları tablosu ve `cv2.Stitcher` çıktısıyla yan yana görsel.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, 4. baskı, Bölüm 12.2–12.3](https://www.imageprocessingplace.com/) — ©
- [Hartley & Zisserman — Multiple View Geometry, Bölüm 4 (DLT, normalizasyon, RANSAC)](https://www.robots.ox.ac.uk/~vgg/hzbook/) — ©
- [Fischler & Bolles — Random Sample Consensus (CACM 1981)](https://doi.org/10.1145/358669.358692) — ©
- [Bolme, Beveridge, Draper, Lui — Visual Object Tracking using Adaptive Correlation Filters (CVPR 2010)](https://doi.org/10.1109/CVPR.2010.5539960) — ©
- [OpenCV — matchTemplate, findHomography](https://docs.opencv.org/4.x/df/dfb/group__imgproc__object.html) — Apache-2.0